# 🔥 Advanced · LLM agent with tool use

> 🔥 **Advanced / heavy lab.** An LLM decides which tool to call, observes the result, and iterates (ReAct) to solve tasks.
>
> **Runtime → Change runtime type → T4 GPU required.** Est. **10–20 min** including downloads. Built on the official **[transformers (function-calling)](https://huggingface.co/docs/transformers/main/en/chat_extras)** and authored to its documented recipe — **not pre-executed here** (needs a GPU + large downloads). If a step fails, see *Troubleshooting* at the bottom; pin versions as noted.

The LLM-powered version of the self-contained **agent harness** (track AG) — drop it into that harness to score it.

## Compute · storage · time

| Resource | Demo (this notebook, free Colab T4) | Full / production run |
|---|---|---|
| **GPU** | T4 16 GB — 1.5B model inference | 24 GB for a 7–14B agent |
| **Storage** | model ~ 3 GB | models 14–30 GB; tool / state stores small |
| **Time** | seconds–min / task (multiple LLM calls) | depends on steps / tools; serve via vLLM for throughput |

**Full pipeline (scale-up):** wrap in LangGraph / smolagents; serve the LLM with vLLM (serving lab).

> Rough estimates — real numbers depend on hardware, data size and library versions.

In [ ]:
!nvidia-smi -L
!pip install -q "transformers>=4.45" accelerate

## 1 · Tools the model may call

In [ ]:
import math
def calculator(expression: str):
    """Evaluate an arithmetic expression."""
    return eval(expression, {"__builtins__": {}}, {"sqrt": math.sqrt})
def word_length(word: str):
    """Return the number of letters in a word."""
    return len(word)
tools = [calculator, word_length]

## 2 · A tool-calling chat model (ReAct loop)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
ck = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(ck)
model = AutoModelForCausalLM.from_pretrained(ck, torch_dtype="auto", device_map="auto")
msgs = [{"role": "user", "content": "What is sqrt(144) plus the number of letters in 'robotics'?"}]
text = tok.apply_chat_template(msgs, tools=tools, add_generation_prompt=True, tokenize=False)
out = model.generate(**tok(text, return_tensors="pt").to(model.device), max_new_tokens=256)
print(tok.decode(out[0], skip_special_tokens=True))

## 3 · Execute the tool call(s) and feed results back
Parse the model's tool call, run the Python function, append the result as a `tool` message, and generate again until it answers. (The chat template handles the tool-call format.)

In [ ]:
# Minimal loop sketch: parse tool_calls from the model output, run the matching
# Python function, append {"role": "tool", "content": str(result)} and re-generate.
# Plug this agent into AG_agent_harness.evaluate(...) to score it on the task suite.

## Save & persist your results
This pipeline writes its checkpoints, metrics/logs and outputs to the run/output directory shown above (e.g. `output/`, `outputs/`, `logdir/`, `experiments/`, or the trainer's `output_dir`). **Colab sessions are ephemeral** — to keep them, either mount Drive and copy the folder (`from google.colab import drive; drive.mount('/content/drive')`) or zip + download it (`import shutil; shutil.make_archive('run','zip','OUTPUT_DIR')` then `from google.colab import files; files.download('run.zip')`). The 🤗 Trainer labs also support `trainer.push_to_hub()`. To publish any output folder as a **model repo** (then group repos into a **Collection** on your profile): `from huggingface_hub import HfApi; HfApi().upload_folder(folder_path="OUTPUT_DIR", repo_id="<you>/ropedia-<lab>")`.

## How this links to tracks A–D
An LLM agent ties the stack together:
- **C · Egocentric** an assistant that reasons over what you see · **D · Scene / world** an embodied agent that acts · **LM** the policy is a language model.

## Next steps
- Use a framework — **LangGraph**, **smolagents**, or **transformers** agents — for robust parsing, memory and multi-step planning.
- Add retrieval (the RAG lab) as a tool; give it environment actions for an embodied agent (track D).